# Lab 1: AXI-Lite on AUP-ZU3

Run this notebook in Jupyter on a 4 GB AUP-ZU3 running PYNQ 3.1.1.
On the build computer, run `make bitstream TARGET=zu3` from `project_files/lab1`.
Upload `build/zu3/mul.bit`, `build/zu3/mul.hwh`, and this notebook into the same board-side directory.
Both overlay files must come from the same build and have the same basename.

The HLS function takes a signed 32-bit integer and returns twice its value.
The notebook is supplied without saved outputs; run it to see the results on your board.


In [ ]:
from pathlib import Path
from time import monotonic, sleep
from pynq import Overlay

bitfile = Path("mul.bit").resolve()
if not bitfile.is_file() or not bitfile.with_suffix(".hwh").is_file():
    raise FileNotFoundError("Upload matching mul.bit and mul.hwh beside this notebook")

ol = Overlay(str(bitfile))
mul = ol.mul_0
print(f"mul_0 base address: {ol.ip_dict['mul_0']['phys_addr']:#x}")


## Inspect the registers

PYNQ obtains the accelerator's base address and register descriptions from `mul.hwh`.
The generated `xmul_hw.h` and `mul_control_s_axi.v` describe the same offsets.
For this source, `CTRL` is at `0x00`, `out_r` is at `0x10`, `out_r_ctrl` is at `0x14`, and `in_r` is at `0x20`.
If you change the function arguments, inspect the regenerated register map before changing the code below.


In [ ]:
print(mul.register_map)
for name, register in ol.ip_dict["mul_0"]["registers"].items():
    print(f"{name}: offset {register['address_offset']:#x}")


## Start the accelerator and wait for AP_DONE

Writing bit 0 of `CTRL` starts one invocation.
Bit 1 is `AP_DONE` and clears when the control register is read.
The helper polls that bit, checks the output-valid bit, and then reads the result.
Negative inputs are written using their 32-bit two's-complement representation.
The output is converted back to a signed Python integer.
Inputs are restricted so doubling them cannot overflow the signed C `int` used in `mul.cpp`.


In [ ]:
CTRL = 0x00
OUT = 0x10
OUT_VALID = 0x14
IN = 0x20


def twice(value, timeout=1.0):
    if not isinstance(value, int) or not -(1 << 30) <= value < (1 << 30):
        raise ValueError("Input must be an integer in [-2**30, 2**30 - 1]")
    if timeout <= 0:
        raise ValueError("timeout must be positive")
    if not (mul.read(CTRL) & 0x4):
        raise RuntimeError("Accelerator is busy; wait for it or reload the overlay")
    mul.read(OUT_VALID)  # Clear a previous output-valid indication.
    mul.write(IN, value & 0xFFFFFFFF)
    mul.write(CTRL, 0x1)
    deadline = monotonic() + timeout
    while not (mul.read(CTRL) & 0x2):
        if monotonic() >= deadline:
            raise TimeoutError("mul_0 did not finish; check the PL clock and reset")
        sleep(0.0001)
    if not (mul.read(OUT_VALID) & 0x1):
        raise RuntimeError("mul_0 finished without a valid output")
    result = mul.read(OUT)
    return result - (1 << 32) if result & (1 << 31) else result


In [ ]:
result = twice(5)
assert result == 10, result
print(f"2 * 5 = {result}")


In [ ]:
for value in [0, 1, -1, 5, -7, 12345, -(1 << 30), (1 << 30) - 1]:
    result = twice(value)
    assert result == 2 * value, (value, result)
    print(f"2 * {value} = {result}")


## If a check fails

- A missing `mul_0` or register map usually means the uploaded HWH does not match the bitstream.
- A timeout calls for checking the processor-to-PL AXI connection, the 100 MHz PL clock, and the active-low reset in Vivado.
- After a timeout, reload the overlay before starting another call.
- Run this notebook on the board's PYNQ Python kernel; the build computer's `pp4fpga` environment does not provide FPGA access.

Once the assertions pass on your board, save the notebook with its actual outputs.
